# Process Raw Chamber Data

Load raw chamber measurements, map user-specific column names into the FCS schema, preview the data, and run standard FCS processing with optional MCMC. This notebook supports the project JSON folder format and generic CSV files.


## Setup


In [1]:
import pathlib
import sys
import urllib.parse

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

CWD = pathlib.Path.cwd().resolve()
for candidate in [CWD, *CWD.parents]:
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate
        break
else:
    REPO_ROOT = CWD

NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "processing"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from soilgasflux_fcs import Multiprocessor, json_reader

DEFAULT_OUTPUT_DIR = NOTEBOOK_DIR / "output"
DEFAULT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_COLUMNS = [
    "datetime",
    "id",
    "timedelta",
    "k30_co2",
    "si_temperature",
    "si_humidity",
    "bmp_pressure",
]


## Loading And Normalization Helpers


In [2]:
def sanitize_input_path(input_path):
    text = str(input_path or "").strip()
    if not text:
        raise FileNotFoundError("Input path is empty. Paste a file or folder path before loading.")

    if text.startswith(("Path(", "pathlib.Path(")) and text.endswith(")"):
        text = text[text.find("(") + 1:-1].strip()

    for _ in range(2):
        if len(text) >= 2 and text[0] == text[-1] and text[0] in {"'", '"'}:
            text = text[1:-1].strip()

    if text.endswith("()"):
        text = text[:-2].strip()

    parsed = urllib.parse.urlparse(text)
    if parsed.scheme == "file":
        text = urllib.parse.unquote(parsed.path)

    if text.startswith("Users/"):
        text = "/" + text

    return pathlib.Path(text).expanduser()


def find_input_files(input_path, pattern="*.csv"):
    path = sanitize_input_path(input_path)
    if path.is_file():
        return [path]
    if path.is_dir():
        files = sorted(file for file in path.glob(pattern) if file.is_file())
        if files:
            return files
        raise FileNotFoundError(
            f"No CSV files matched pattern {pattern!r} in {path}. "
            "Check the input mode, folder, and CSV pattern."
        )
    raise FileNotFoundError(f"Input path does not exist: {path}")


def find_json_files(folder_path, min_size_bytes=5000):
    folder = sanitize_input_path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"JSON folder does not exist: {folder}")
    if not folder.is_dir():
        raise NotADirectoryError(f"JSON input must be a folder, not a file: {folder}")

    all_json = sorted(file for file in folder.rglob("*.json") if file.is_file())
    usable = [file for file in all_json if file.stat().st_size >= min_size_bytes]
    ignored = [file for file in all_json if file.stat().st_size < min_size_bytes]

    if not all_json:
        raise FileNotFoundError(f"No JSON files found in this folder: {folder}")
    if not usable:
        raise FileNotFoundError(
            f"No usable JSON files found in {folder}. Found {len(all_json)} JSON file(s), "
            f"but all were smaller than {min_size_bytes} bytes."
        )
    return usable, ignored


def load_json_folder(folder_path, min_size_bytes=5000):
    folder = sanitize_input_path(folder_path)
    files, ignored = find_json_files(folder, min_size_bytes=min_size_bytes)
    initializer = json_reader.Initializer(folder)
    try:
        df = initializer.prepare_rawdata()
    except ValueError as exc:
        if "No objects to concatenate" in str(exc) or "no objects to concatenate" in str(exc):
            raise FileNotFoundError(
                f"No usable JSON raw-data files were loaded from {folder}. "
                "Check that the folder contains FCS JSON files with a raw_data section."
            ) from exc
        raise
    if df.empty:
        raise FileNotFoundError(f"JSON files were found in {folder}, but no rows were loaded.")
    df.attrs["source_path"] = str(folder)
    df.attrs["source_files"] = [str(file) for file in files]
    df.attrs["ignored_small_json_files"] = [str(file) for file in ignored]
    return df[CANONICAL_COLUMNS + [c for c in df.columns if c not in CANONICAL_COLUMNS]]


def load_csv_files(input_path, pattern="*.csv", delimiter=","):
    files = find_input_files(input_path, pattern=pattern)
    frames = []
    for file in files:
        df = pd.read_csv(file, sep=delimiter)
        df["__source_file"] = file.stem
        frames.append(df)
    if not frames:
        raise FileNotFoundError(
            f"No CSV files could be read from {sanitize_input_path(input_path)} with pattern {pattern!r}."
        )
    raw = pd.concat(frames, ignore_index=True)
    raw.attrs["source_files"] = [str(file) for file in files]
    return raw


def guess_column(columns, candidates):
    normalized = {str(col).lower().strip(): col for col in columns}
    for candidate in candidates:
        key = candidate.lower().strip()
        if key in normalized:
            return normalized[key]
    for col in columns:
        lower = str(col).lower()
        if any(candidate.lower() in lower for candidate in candidates):
            return col
    return ""


def _require_or_fill(raw_df, source_col, output_col, fill_missing_environment, default_value):
    if source_col:
        return raw_df[source_col]
    if fill_missing_environment:
        return default_value
    raise ValueError(
        f"Missing mapping for {output_col}. Select a source column or enable constant-fill mode."
    )


def normalize_csv_dataframe(
    raw_df,
    *,
    co2_col,
    timestamp_col="",
    elapsed_col="",
    id_col="",
    pressure_col="",
    temperature_col="",
    humidity_col="",
    pressure_unit="Pa",
    fill_missing_environment=False,
    default_pressure_pa=101325.0,
    default_temperature_c=20.0,
    default_humidity_percent=70.0,
):
    if not co2_col:
        raise ValueError("CO2 column is required.")
    if not timestamp_col and not elapsed_col:
        raise ValueError("Select either a timestamp column or an elapsed-seconds column.")

    df = pd.DataFrame()
    if id_col:
        df["id"] = raw_df[id_col].astype(str)
    elif "__source_file" in raw_df.columns:
        df["id"] = raw_df["__source_file"].astype(str)
    else:
        df["id"] = "measurement_001"

    df["k30_co2"] = pd.to_numeric(raw_df[co2_col], errors="coerce")

    if timestamp_col:
        df["datetime"] = pd.to_datetime(raw_df[timestamp_col], errors="coerce")
        df["timedelta"] = (
            df.groupby("id")["datetime"]
            .transform(lambda values: (values - values.min()).dt.total_seconds())
            .astype("float")
        )
    else:
        df["timedelta"] = pd.to_numeric(raw_df[elapsed_col], errors="coerce")
        base = pd.Timestamp("2000-01-01")
        offsets = {measurement_id: n for n, measurement_id in enumerate(df["id"].drop_duplicates())}
        df["datetime"] = [
            base + pd.Timedelta(days=offsets[measurement_id]) + pd.Timedelta(seconds=float(seconds))
            if pd.notna(seconds) else pd.NaT
            for measurement_id, seconds in zip(df["id"], df["timedelta"])
        ]

    pressure = _require_or_fill(
        raw_df,
        pressure_col,
        "bmp_pressure",
        fill_missing_environment,
        default_pressure_pa,
    )
    pressure = pd.to_numeric(pressure, errors="coerce")
    if pressure_col and pressure_unit == "kPa":
        pressure = pressure * 1000.0
    df["bmp_pressure"] = pressure

    df["si_temperature"] = pd.to_numeric(
        _require_or_fill(
            raw_df,
            temperature_col,
            "si_temperature",
            fill_missing_environment,
            default_temperature_c,
        ),
        errors="coerce",
    )
    df["si_humidity"] = pd.to_numeric(
        _require_or_fill(
            raw_df,
            humidity_col,
            "si_humidity",
            fill_missing_environment,
            default_humidity_percent,
        ),
        errors="coerce",
    )

    df = df.sort_values(["id", "datetime", "timedelta"]).reset_index(drop=True)
    return df[CANONICAL_COLUMNS]


def validate_fcs_dataframe(df):
    missing = [column for column in CANONICAL_COLUMNS if column not in df.columns]
    null_counts = df[CANONICAL_COLUMNS].isna().sum().to_dict() if not missing else {}
    valid = not missing and all(count == 0 for count in null_counts.values())
    return {
        "valid": valid,
        "missing_columns": missing,
        "null_counts": null_counts,
        "n_measurements": int(df["id"].nunique()) if "id" in df.columns else 0,
        "n_rows": int(len(df)),
    }


def summarize_measurements(df):
    summary = (
        df.groupby("id")
        .agg(
            n_rows=("timedelta", "size"),
            start=("datetime", "min"),
            end=("datetime", "max"),
            duration_s=("timedelta", "max"),
            min_co2=("k30_co2", "min"),
            max_co2=("k30_co2", "max"),
        )
        .reset_index()
    )
    return summary


def plot_measurement(df, measurement_id=None):
    measurement_id = measurement_id or df["id"].iloc[0]
    selected = df[df["id"].astype(str) == str(measurement_id)]
    fig, ax = plt.subplots(figsize=(7, 3.5), dpi=120)
    ax.plot(selected["timedelta"], selected["k30_co2"], color="#1f77b4", linewidth=1.8)
    ax.scatter(selected["timedelta"], selected["k30_co2"], color="#1f77b4", s=10, alpha=0.4)
    ax.set_xlabel("Elapsed time [s]")
    ax.set_ylabel(r"$CO_2$ [ppm]")
    ax.set_title(f"Measurement: {measurement_id}")
    fig.tight_layout()
    return fig


def run_fcs_processing(
    df,
    *,
    chamber_id,
    output_folder=DEFAULT_OUTPUT_DIR,
    area=314.0,
    volume=6283.0,
    use_mcmc=False,
    n_mc=500,
    sensor_precision=None,
):
    validation = validate_fcs_dataframe(df)
    if not validation["valid"]:
        raise ValueError(f"Dataframe is not ready for FCS processing: {validation}")

    output_folder = pathlib.Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    processor = Multiprocessor()
    metadata = {"area": area, "volume": volume}
    if use_mcmc:
        return processor.run_MC(
            df=df,
            chamber_id=chamber_id,
            output_folder=str(output_folder),
            save_netcdf=True,
            sensor_precision=sensor_precision,
            n_MC=n_mc,
            metadata=metadata,
        )
    return processor.run(
        df=df,
        chamber_id=chamber_id,
        output_folder=str(output_folder),
        metadata=metadata,
    )


## Plain Python Example

Use these calls directly in scripts, or use the widget section below for an interactive workflow. Paths must be Python strings when used in code cells.


In [3]:
# JSON folder example
# df = load_json_folder("/Users/alexnaokiasatokobayashi/Downloads/test_folder/2026-04-17/")
# display(df.head())
# display(summarize_measurements(df).head())

# CSV example
# raw = load_csv_files("path/to/csv_folder", pattern="*.csv", delimiter=",")
# df = normalize_csv_dataframe(
#     raw,
#     timestamp_col="timestamp",
#     co2_col="co2",
#     pressure_col="pressure",
#     pressure_unit="Pa",
#     temperature_col="temp",
#     humidity_col="rh",
#     fill_missing_environment=False,
# )
# display(summarize_measurements(df).head())


## Interactive Workflow


In [4]:
import ipywidgets as widgets
from IPython.display import clear_output, display

state = {"raw": None, "df": None, "updating_controls": False}
style = {"description_width": "160px"}
wide = widgets.Layout(width="780px")
path_layout = widgets.Layout(width="780px", height="74px")
medium = widgets.Layout(width="380px")

input_mode_widget = widgets.Dropdown(options=["CSV", "JSON folder"], value="CSV", description="Input mode", style=style, layout=medium)
input_path_widget = widgets.Textarea(
    value="",
    description="Input path",
    placeholder="Paste a file or folder path here",
    continuous_update=False,
    style=style,
    layout=path_layout,
)
current_path_widget = widgets.HTML(value="<b>Current path:</b> none")
pattern_widget = widgets.Text(value="*.csv", description="CSV pattern", continuous_update=False, style=style, layout=medium)
delimiter_widget = widgets.Text(value=",", description="CSV delimiter", continuous_update=False, style=style, layout=medium)

unloaded_option = [("Load input first", "")]
id_col_widget = widgets.Dropdown(options=unloaded_option, description="Measurement id", disabled=True, style=style, layout=medium)
timestamp_col_widget = widgets.Dropdown(options=unloaded_option, description="Timestamp", disabled=True, style=style, layout=medium)
elapsed_col_widget = widgets.Dropdown(options=unloaded_option, description="Elapsed seconds", disabled=True, style=style, layout=medium)
co2_col_widget = widgets.Dropdown(options=unloaded_option, description="CO2", disabled=True, style=style, layout=medium)
pressure_col_widget = widgets.Dropdown(options=unloaded_option, description="Pressure", disabled=True, style=style, layout=medium)
temperature_col_widget = widgets.Dropdown(options=unloaded_option, description="Temperature", disabled=True, style=style, layout=medium)
humidity_col_widget = widgets.Dropdown(options=unloaded_option, description="Humidity", disabled=True, style=style, layout=medium)
pressure_unit_widget = widgets.Dropdown(options=["Pa", "kPa"], value="Pa", description="Pressure unit", style=style, layout=medium)
fill_missing_widget = widgets.Checkbox(value=False, description="Allow constant environmental defaults", indent=False, layout=wide)
default_pressure_widget = widgets.FloatText(value=101325.0, description="Default pressure [Pa]", style=style, layout=medium)
default_temperature_widget = widgets.FloatText(value=20.0, description="Default temp [C]", style=style, layout=medium)
default_humidity_widget = widgets.FloatText(value=70.0, description="Default RH [%]", style=style, layout=medium)

mapping_status_widget = widgets.HTML(value="Load an input before selecting column mappings.")
process_scope_widget = widgets.Dropdown(
    options=[
        ("Whole folder", "all"),
        ("Specific ID", "id"),
        ("Specific date", "date"),
    ],
    value="all",
    description="Processing scope",
    style=style,
    layout=widgets.Layout(width="430px"),
)
measurement_widget = widgets.Dropdown(options=[], description="Specific id", style=style, layout=widgets.Layout(width="430px", display="none"))
process_date_widget = widgets.Dropdown(options=[], description="Specific date", style=style, layout=widgets.Layout(width="430px", display="none"))
process_status_widget = widgets.HTML(value="Normalize the loaded data to populate processing targets.")
chamber_id_widget = widgets.Text(value="analysis", description="Chamber id", continuous_update=False, style=style, layout=medium)
output_folder_widget = widgets.Textarea(
    value=str(DEFAULT_OUTPUT_DIR),
    description="Output folder",
    placeholder="Paste an output folder path here",
    continuous_update=False,
    style=style,
    layout=path_layout,
)
area_widget = widgets.FloatText(value=314.0, description="Area [cm2]", style=style, layout=medium)
volume_widget = widgets.FloatText(value=6283.0, description="Volume [cm3]", style=style, layout=medium)
use_mcmc_widget = widgets.Checkbox(value=False, description="Run MCMC", indent=False, layout=medium)
n_mc_widget = widgets.IntText(value=500, description="n_MC", style=style, layout=medium)
sensor_precision_widget = widgets.FloatText(value=np.nan, description="Sensor precision", style=style, layout=medium)

load_button = widgets.Button(description="Load input", button_style="primary")
normalize_button = widgets.Button(description="Normalize + preview", button_style="info")
process_button = widgets.Button(description="Run FCS", button_style="success")
output = widgets.Output()


column_widgets = [
    id_col_widget,
    timestamp_col_widget,
    elapsed_col_widget,
    co2_col_widget,
    pressure_col_widget,
    temperature_col_widget,
    humidity_col_widget,
]
normalization_widgets = column_widgets + [
    pressure_unit_widget,
    fill_missing_widget,
    default_pressure_widget,
    default_temperature_widget,
    default_humidity_widget,
]


def _clear_column_options():
    state["updating_controls"] = True
    try:
        for widget in column_widgets:
            widget.options = unloaded_option
            widget.value = ""
            widget.disabled = True
    finally:
        state["updating_controls"] = False
    mapping_status_widget.value = "Load an input before selecting column mappings."


def _set_column_options(columns):
    columns = list(columns)
    options = [""] + columns
    state["updating_controls"] = True
    try:
        for widget in column_widgets:
            widget.options = options
            widget.disabled = False

        id_col_widget.value = guess_column(columns, ["id", "measurement", "measurement_id", "chamber"])
        timestamp_col_widget.value = guess_column(columns, ["timestamp", "datetime", "datetime_utc", "time"])
        elapsed_col_widget.value = guess_column(columns, ["timedelta", "elapsed", "seconds", "time_s"])
        co2_col_widget.value = guess_column(columns, ["co2", "k30_co2", "co2_ppm"])
        pressure_col_widget.value = guess_column(columns, ["pressure", "bmp_pressure", "chamber_p"])
        temperature_col_widget.value = guess_column(columns, ["temperature", "temp", "si_temperature", "chamber_t"])
        humidity_col_widget.value = guess_column(columns, ["humidity", "rh", "si_humidity"])
    finally:
        state["updating_controls"] = False
    mapping_status_widget.value = f"Detected {len(columns)} columns. Review the mappings, then normalize the data."


def _set_processing_scope_visibility(change=None):
    measurement_widget.layout.display = "" if process_scope_widget.value == "id" else "none"
    process_date_widget.layout.display = "" if process_scope_widget.value == "date" else "none"


def _refresh_processing_targets(df=None):
    state["updating_controls"] = True
    try:
        if df is None or df.empty:
            measurement_widget.options = []
            process_date_widget.options = []
            process_status_widget.value = "Normalize the loaded data to populate processing targets."
        else:
            measurement_widget.options = sorted(df["id"].astype(str).unique())
            process_date_widget.options = sorted(df["datetime"].dt.strftime("%Y-%m-%d").unique())
            process_status_widget.value = (
                f"Ready: {len(df)} rows, {df['id'].nunique()} IDs, "
                f"{df['datetime'].dt.date.nunique()} dates."
            )
    finally:
        state["updating_controls"] = False
    _set_processing_scope_visibility()


def _select_processing_dataframe(df, scope, measurement_id=None, date=None):
    if scope == "all":
        selected = df.copy()
    elif scope == "id":
        if not measurement_id:
            raise ValueError("Select a measurement ID before running FCS.")
        selected = df[df["id"].astype(str) == str(measurement_id)].copy()
    elif scope == "date":
        if not date:
            raise ValueError("Select a date before running FCS.")
        selected = df[df["datetime"].dt.strftime("%Y-%m-%d") == str(date)].copy()
    else:
        raise ValueError(f"Unknown processing scope: {scope!r}")

    if selected.empty:
        raise ValueError("The selected processing scope contains no rows. Normalize again and choose an available target.")
    return selected


def _processing_scope_label(scope, measurement_id=None, date=None):
    if scope == "id":
        return f"Specific ID: {measurement_id}"
    if scope == "date":
        return f"Specific date: {date}"
    return "Whole folder"


def _invalidate_normalized_data(change=None):
    if state["updating_controls"]:
        return
    state["df"] = None
    _refresh_processing_targets()
    if state["raw"] is not None:
        mapping_status_widget.value = "Mappings changed. Normalize again before running FCS."


def _invalidate_loaded_input(change=None):
    if state["updating_controls"]:
        return
    state["raw"] = None
    state["df"] = None
    _clear_column_options()
    _refresh_processing_targets()


def _show_error(exc):
    print(f"{type(exc).__name__}: {exc}")


def on_load_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            state["raw"] = None
            state["df"] = None
            _clear_column_options()
            _refresh_processing_targets()
            path = sanitize_input_path(input_path_widget.value)
            current_path_widget.value = f"<b>Current path:</b> {path}"

            if input_mode_widget.value == "JSON folder":
                files, ignored = find_json_files(path)
                print(f"Matched {len(files)} usable JSON file(s) in {path}.")
                if ignored:
                    print(f"Ignored {len(ignored)} small JSON file(s).")
                raw = load_json_folder(path)
                state["raw"] = raw
                _set_column_options(raw.columns)
                display(raw.head())
                display(summarize_measurements(raw))
                print(f"Detected {len(raw.columns)} columns and {len(raw)} rows.")
                print("Review the JSON mappings in the Map Columns tab, then normalize the data.")
                return

            files = find_input_files(path, pattern=pattern_widget.value)
            print(f"Matched {len(files)} CSV file(s) in {path}.")
            raw = load_csv_files(
                path,
                pattern=pattern_widget.value,
                delimiter=delimiter_widget.value,
            )
            state["raw"] = raw
            state["df"] = None
            _set_column_options(raw.columns)
            display(raw.head())
            print(f"Detected {len(raw.columns)} columns and {len(raw)} rows.")
        except Exception as exc:
            _show_error(exc)


def on_normalize_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            if state["raw"] is None:
                raise ValueError("Load input before normalizing.")
            df = normalize_csv_dataframe(
                state["raw"],
                id_col=id_col_widget.value,
                timestamp_col=timestamp_col_widget.value,
                elapsed_col=elapsed_col_widget.value,
                co2_col=co2_col_widget.value,
                pressure_col=pressure_col_widget.value,
                temperature_col=temperature_col_widget.value,
                humidity_col=humidity_col_widget.value,
                pressure_unit=pressure_unit_widget.value,
                fill_missing_environment=fill_missing_widget.value,
                default_pressure_pa=default_pressure_widget.value,
                default_temperature_c=default_temperature_widget.value,
                default_humidity_percent=default_humidity_widget.value,
            )
            state["df"] = df

            validation = validate_fcs_dataframe(df)
            _refresh_processing_targets(df)
            mapping_status_widget.value = "Normalization complete. Processing targets are ready."
            display(df.head())
            display(summarize_measurements(df))
            display(validation)
            if validation["valid"] and len(measurement_widget.options):
                display(plot_measurement(df, measurement_widget.value))
        except Exception as exc:
            _show_error(exc)


def on_process_clicked(_):
    with output:
        clear_output(wait=True)
        try:
            if state["df"] is None:
                raise ValueError("Normalize data before running FCS.")
            processing_df = _select_processing_dataframe(
                state["df"],
                process_scope_widget.value,
                measurement_id=measurement_widget.value,
                date=process_date_widget.value,
            )
            validation = validate_fcs_dataframe(processing_df)
            if not validation["valid"]:
                raise ValueError(f"Selected data is not ready for FCS processing: {validation}")
            selected_dates = sorted(processing_df["datetime"].dt.strftime("%Y-%m-%d").unique())
            print(
                f"Processing scope: {_processing_scope_label(process_scope_widget.value, measurement_widget.value, process_date_widget.value)}"
            )
            print(
                f"Selected {len(processing_df)} rows, {processing_df['id'].nunique()} IDs, "
                f"dates: {', '.join(selected_dates)}"
            )
            sensor_precision = None if np.isnan(sensor_precision_widget.value) else sensor_precision_widget.value
            result = run_fcs_processing(
                processing_df,
                chamber_id=chamber_id_widget.value,
                output_folder=output_folder_widget.value,
                area=area_widget.value,
                volume=volume_widget.value,
                use_mcmc=use_mcmc_widget.value,
                n_mc=n_mc_widget.value,
                sensor_precision=sensor_precision,
            )
            print("FCS processing complete.")
            print(f"Output folder: {output_folder_widget.value}")
            display(result)
        except Exception as exc:
            _show_error(exc)


load_button.on_click(on_load_clicked)
normalize_button.on_click(on_normalize_clicked)
process_button.on_click(on_process_clicked)
process_scope_widget.observe(_set_processing_scope_visibility, names="value")
for widget in normalization_widgets:
    widget.observe(_invalidate_normalized_data, names="value")
for widget in [input_mode_widget, input_path_widget, pattern_widget, delimiter_widget]:
    widget.observe(_invalidate_loaded_input, names="value")

input_controls = widgets.VBox([
    input_mode_widget,
    input_path_widget,
    current_path_widget,
    widgets.HBox([pattern_widget, delimiter_widget]),
    widgets.HBox([load_button, normalize_button]),
])

mapping_controls = widgets.VBox([
    mapping_status_widget,
    widgets.HBox([id_col_widget, timestamp_col_widget]),
    widgets.HBox([elapsed_col_widget, co2_col_widget]),
    widgets.HBox([pressure_col_widget, pressure_unit_widget]),
    widgets.HBox([temperature_col_widget, humidity_col_widget]),
    fill_missing_widget,
    widgets.HBox([default_pressure_widget, default_temperature_widget]),
    default_humidity_widget,
])

processing_controls = widgets.VBox([
    process_status_widget,
    widgets.HBox([process_scope_widget, measurement_widget, process_date_widget]),
    chamber_id_widget,
    output_folder_widget,
    widgets.HBox([area_widget, volume_widget]),
    widgets.HBox([use_mcmc_widget, n_mc_widget, sensor_precision_widget]),
    process_button,
])

tabs = widgets.Tab(children=[input_controls, mapping_controls, processing_controls])
tabs.set_title(0, "Load")
tabs.set_title(1, "Map Columns")
tabs.set_title(2, "Process")
display(tabs, output)


Output()